# Stage 1 - Gemma QA Fine-tuning

This notebook prepares the Colab runtime, inspects the GPU environment, uploads a training zip, and fine-tunes a Gemma-family model with checkpoint/resume support.

If you run this from VS Code with the Google Colab extension:
- connect the notebook to `Colab > Auto Connect`
- optionally use `Upload to Colab` on this repository if you want to run your local uncommitted files
- then execute the setup cells below

In [ ]:
#@title 1. Prepare repository on the Colab runtime
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Yongjooon/QA-AI-FineTunning.git"
REPO_NAME = "QA-AI-FineTunning"
DEFAULT_REPO_DIR = f"/content/{REPO_NAME}"

candidates = []
env_repo_dir = os.environ.get("QA_FINETUNE_REPO_DIR", "").strip()
if env_repo_dir:
    candidates.append(Path(env_repo_dir))
candidates.append(Path.cwd())
candidates.append(Path(DEFAULT_REPO_DIR))
candidates.extend(Path("/content").glob(f"{REPO_NAME}*"))

repo_dir = None
for candidate in candidates:
    if (candidate / "pyproject.toml").exists():
        repo_dir = candidate
        break

if repo_dir is None:
    repo_dir = Path(DEFAULT_REPO_DIR)
    print(f"Repository not found on runtime. Cloning from {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
else:
    print(f"Using repository on runtime: {repo_dir}")

os.chdir(repo_dir)
print("Current working directory:", os.getcwd())

In [ ]:
#@title 2. Install dependencies
!pip -q install -U pip
!pip -q install -r requirements-colab.txt
!pip -q install -e .

In [ ]:
#@title 3. Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 4. Optional Hugging Face login if the model is gated
# Uncomment if needed.
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
#@title 5. Training parameters
MODEL_NAME = "google/gemma-3-4b-it"  # Change to the exact Gemma checkpoint you want
PERSIST_ROOT = "/content/drive/MyDrive/qa_ai_finetuning"  # Persistent output location
RUN_NAME = "gemma_qa_lora_run_01"
RESUME_MODE = "auto"  # auto | never | path
RESUME_CHECKPOINT = ""  # Only used when RESUME_MODE == 'path'
NUM_TRAIN_EPOCHS = 3.0
LEARNING_RATE = 2e-4
SAVE_STEPS = 50
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 4
MAX_TRAIN_SAMPLES = 0  # 0 keeps all records

In [ ]:
#@title 6. Inspect Colab runtime
import json
from qafinetune.runtime import detect_runtime, suggest_training_preset

runtime_profile = detect_runtime()
print(json.dumps(runtime_profile, indent=2, ensure_ascii=False))
print("\nSuggested preset:")
try:
    print(json.dumps(suggest_training_preset(runtime_profile), indent=2, ensure_ascii=False))
except RuntimeError as exc:
    print(f"Preset unavailable yet: {exc}")
    print("Switch the Colab runtime to GPU and rerun this cell before training.")

In [ ]:
#@title 7. Upload training zip
from google.colab import files
uploaded = files.upload()
TRAIN_ZIP_PATH = next(iter(uploaded.keys()))
print("Uploaded:", TRAIN_ZIP_PATH)

In [ ]:
#@title 8. Run training
import subprocess

command = [
    "python", "-m", "qafinetune.train",
    "--model_name", MODEL_NAME,
    "--train_zip", TRAIN_ZIP_PATH,
    "--output_root", PERSIST_ROOT,
    "--run_name", RUN_NAME,
    "--resume_mode", RESUME_MODE,
    "--resume_checkpoint", RESUME_CHECKPOINT,
    "--num_train_epochs", str(NUM_TRAIN_EPOCHS),
    "--learning_rate", str(LEARNING_RATE),
    "--save_steps", str(SAVE_STEPS),
    "--logging_steps", str(LOGGING_STEPS),
    "--save_total_limit", str(SAVE_TOTAL_LIMIT),
    "--max_train_samples", str(MAX_TRAIN_SAMPLES),
]
print(" ".join(command))
subprocess.run(command, check=True)

In [ ]:
#@title 9. Inspect saved run state
import json
from pathlib import Path

run_state_path = Path(PERSIST_ROOT) / "train_runs" / RUN_NAME / "run_state.json"
print(run_state_path)
print(json.dumps(json.loads(run_state_path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))